# Tugas 9 - Mini Klasifikasi: Preferensi Belanja Online vs Offline

**Kelompok :** 3   
**Anggota 1 :** Syarifah Nesia (2024091017) - TEKNIK SIPIL   
**Anggota 2 :** Panji Kurnia Akbar (2024081024) - SISTEM INFORMASI    
**Mata Kuliah:** Introduction to AI

## 1. Import Library dan Konfigurasi Visual
Bagian ini memuat:
- Import library utama.
- Fungsi bantu untuk menampilkan tabel yang rapi dan bisa digeser.

## 2. Load Dataset dan Tinjauan Awal Data
Bagian ini memuat:
- Membaca dataset dan memilih 20 data pertama.
- Menentukan kolom fitur dan label.
- Menggabungkan fitur menjadi teks dan melakukan pembersihan sederhana.
- Menampilkan contoh sebelum dan sesudah preprocessing.

In [13]:
import re
from pathlib import Path

import pandas as pd
from IPython.display import display, HTML, Markdown
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

DATA_PATH = Path("Survei Perilaku Konsumen_ Analisis Preferensi Belanja Online vs. Offline Berbasis AI (Jawaban) - Form Responses 1.csv")

def render_scroll_table(df_to_show: pd.DataFrame, title: str) -> None:
    display(Markdown(f"### {title}"))
    html_table = df_to_show.to_html(index=True)
    table_html = f"""
<div style="overflow-x:auto; border:1px solid #e0e0e0; padding:8px; border-radius:6px;">
  <style>
    table.dataframe {{
      border-collapse: collapse;
      width: max-content;
      max-width: 100%;
      font-family: sans-serif;
      font-size: 13px;
    }}
    table.dataframe th, table.dataframe td {{
      border: 1px solid #dddddd;
      padding: 6px 8px;
      text-align: left;
      vertical-align: top;
      white-space: nowrap;
    }}
  </style>
  {html_table}
</div>
"""
    display(HTML(table_html))

# Konfigurasi tampilan dataframe
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)

### 2.1 Load dataset dan filter nama
- Membaca dataset.
- Menghapus nama yang tidak jelas.
- Menentukan kolom label dan fitur.

In [14]:
df = pd.read_csv(DATA_PATH)
df.columns = [col.strip() for col in df.columns]  # rapikan spasi nama kolom

name_col = "Nama Lengkap"
invalid_names = {"Meta", "2394"}
df_filtered = df[~df[name_col].astype(str).str.strip().isin(invalid_names)].copy()
df_sample = df_filtered.head(20).copy()

label_col = "Secara keseluruhan, dalam kondisi normal, platform mana yang lebih Anda prioritaskan untuk belanja saat ini?"
freq_col = "Seberapa sering Anda berbelanja kebutuhan melalui platform Online (E-commerce/Sosmed) dalam sebulan terakhir?"
reason_online_col = "Apa faktor utama yang membuat Anda memilih belanja Online?"
reason_offline_col = "Saat berbelanja Offline (Toko Fisik/Mall), apa alasan utama Anda tetap melakukannya?"

df_sample = df_sample[[freq_col, reason_online_col, reason_offline_col, label_col]].dropna()

### 2.2 Tinjauan awal data
- Preview 20 data pertama.
- Daftar kolom/pertanyaan pada dataset.

In [15]:
preview_df = df_filtered.head(20).copy().reset_index(drop=True)
render_scroll_table(preview_df, "Preview Dataset (20 Data Pertama)")

cols_df = pd.DataFrame({"Kolom": df.columns})
render_scroll_table(cols_df, "Daftar Kolom pada Dataset")

### Preview Dataset (20 Data Pertama)

,Timestamp,Nama Lengkap,Jenis Kelamin,Usia,Pekerjaan Utama,Rata-rata Pengeluaran Belanja per Bulan,Seberapa sering Anda berbelanja kebutuhan melalui platform Online (E-commerce/Sosmed) dalam sebulan terakhir?,Kategori produk apa yang paling sering Anda beli secara Online?,Apa faktor utama yang membuat Anda memilih belanja Online?,"Seberapa besar pengaruh ""Ulasan/Review Pembeli"" terhadap keputusan Anda saat belanja Online?","Saat berbelanja Offline (Toko Fisik/Mall), apa alasan utama Anda tetap melakukannya?",Seberapa khawatir Anda terhadap risiko penipuan atau barang tidak sesuai saat belanja Online?,Apakah menurut Anda harga di toko Online jauh lebih murah dibandingkan toko Offline?,Berapa lama waktu yang bersedia Anda tunggu untuk pengiriman barang jika belanja secara Online?,Seberapa penting interaksi langsung dengan penjual/SPG bagi Anda saat berbelanja?,"Secara keseluruhan, dalam kondisi normal, platform mana yang lebih Anda prioritaskan untuk belanja saat ini?"
0,06/04/2026 22:11:43,M.Aldrich Ihza H. (2021) (Semangat ya),Laki-laki,18-25 Tahun,Wiraswasta,< Rp1 Juta,3,Elektronik,Diskon/Promo,5,Bisa mencoba/melihat fisik barang,5,5,2-3 Hari,5,Offline Shopper
1,06/04/2026 22:15:37,Sabrina Qurrota Aina Sunarya,Perempuan,18-25 Tahun,Pelajar/Mahasiswa,< Rp1 Juta,4,Skincare/Kosmetik,"Diskon/Promo, Kemudahan Akses",5,Bisa mencoba/melihat fisik barang,3,5,2-3 Hari,3,Online Shopper
2,06/04/2026 22:16:29,Kruisna Rikzaa Maulana,Laki-laki,18-25 Tahun,Atlet & pelatih Muaythai,< Rp1 Juta,1,Kebutuhan Pokok/Makanan,Kemudahan Akses,3,Bisa mencoba/melihat fisik barang,5,4,2-3 Hari,3,Offline Shopper
3,06/04/2026 22:18:18,LUBNAYYA ARISUNAR,Perempuan,18-25 Tahun,Pelajar/Mahasiswa,Rp3 Juta - Rp5 Juta,5,Skincare/Kosmetik,"Diskon/Promo, Kemudahan Akses, Variasi Produk, Malas Keluar Rumah",4,Bisa mencoba/melihat fisik barang,5,5,2-3 Hari,2,Online Shopper
4,06/04/2026 22:19:30,Amanda Putri Riyanti,Perempuan,18-25 Tahun,Pelajar/Mahasiswa,Rp3 Juta - Rp5 Juta,5,Fashion,"Diskon/Promo, Kemudahan Akses, Malas Keluar Rumah",5,Langsung mendapatkan barang,3,5,4-7 Hari,3,Online Shopper
5,06/04/2026 23:09:52,Halimah Tusa'diyah,Perempuan,18-25 Tahun,Pelajar/Mahasiswa,Rp1 Juta - Rp3 Juta,5,Skincare/Kosmetik,"Diskon/Promo, Kemudahan Akses, Variasi Produk, Malas Keluar Rumah",5,Bisa mencoba/melihat fisik barang,5,5,2-3 Hari,2,Online Shopper
6,07/04/2026 4:39:39,Evania Mandasari Andrianto,Perempuan,18-25 Tahun,Pelajar/Mahasiswa,< Rp1 Juta,3,Kebutuhan Pokok/Makanan,"Diskon/Promo, Kemudahan Akses, Malas Keluar Rumah",4,Bisa mencoba/melihat fisik barang,3,4,2-3 Hari,3,Online Shopper
7,07/04/2026 8:27:14,Nabila Dwi Novianti,Perempuan,18-25 Tahun,Pelajar/Mahasiswa,< Rp1 Juta,3,Skincare/Kosmetik,"Diskon/Promo, Kemudahan Akses, Variasi Produk",5,Bisa mencoba/melihat fisik barang,4,5,2-3 Hari,4,Online Shopper
8,07/04/2026 9:29:39,Maryam Nisfia Erwina,Perempuan,18-25 Tahun,Pelajar/Mahasiswa,< Rp1 Juta,2,Kebutuhan Pokok/Makanan,"Diskon/Promo, Kemudahan Akses, Variasi Produk, Malas Keluar Rumah",4,Bisa mencoba/melihat fisik barang,4,4,4-7 Hari,2,Online Shopper
9,07/04/2026 9:43:44,Arae Mahesa Armera,Laki-laki,18-25 Tahun,Pelajar/Mahasiswa,Rp1 Juta - Rp3 Juta,4,Fashion,"Diskon/Promo, Kemudahan Akses, Variasi Produk",3,Bisa mencoba/melihat fisik barang,2,4,2-3 Hari,3,Online Shopper


### Daftar Kolom pada Dataset

,Kolom
0,Timestamp
1,Nama Lengkap
2,Jenis Kelamin
3,Usia
4,Pekerjaan Utama
5,Rata-rata Pengeluaran Belanja per Bulan
6,Seberapa sering Anda berbelanja kebutuhan melalui platform Online (E-commerce/Sosmed) dalam sebulan terakhir?
7,Kategori produk apa yang paling sering Anda beli secara Online?
8,Apa faktor utama yang membuat Anda memilih belanja Online?
9,"Seberapa besar pengaruh ""Ulasan/Review Pembeli"" terhadap keputusan Anda saat belanja Online?"


### 2.3 Preprocessing teks
- Membersihkan teks (lowercase, hapus karakter non-alfanumerik).
- Menggabungkan fitur menjadi satu teks.
- Menampilkan contoh sebelum dan sesudah preprocessing.

In [16]:
def clean_text(value: str) -> str:
    text = str(value).lower()
    text = re.sub(r"[^0-9a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_sample["text_features"] = (
    "freq_" + df_sample[freq_col].astype(str) + " "
    + df_sample[reason_online_col].astype(str) + " "
    + df_sample[reason_offline_col].astype(str)
)

df_sample["text_features_clean"] = df_sample["text_features"].apply(clean_text)

example_df = pd.DataFrame(
    {"Sebelum Preprocessing": [df_sample["text_features"].iloc[0]], "Sesudah Preprocessing": [df_sample["text_features_clean"].iloc[0]]}
)
render_scroll_table(example_df, "Contoh Preprocessing Teks")

### Contoh Preprocessing Teks

,Sebelum Preprocessing,Sesudah Preprocessing
0,freq_3 Diskon/Promo Bisa mencoba/melihat fisik barang,freq 3 diskon promo bisa mencoba melihat fisik barang


## 3. Pelatihan model dan hasil prediksi
Bagian ini memuat:
- 3.1 Vectorisasi teks dan pembagian data.
- 3.2 Pelatihan model dan prediksi.
- 3.3 Evaluasi akurasi.

In [17]:
# 3.1 Vectorisasi teks dan pembagian data
X = df_sample["text_features_clean"]
y = df_sample[label_col].astype(str)

vectorizer = CountVectorizer()
X_vector = vectorizer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_vector, y, test_size=0.2, random_state=42
 )

### 3.2 Pelatihan model dan prediksi

In [18]:
model = MultinomialNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

results = pd.DataFrame({
    "Label Asli": y_test.values,
    "Prediksi Model": y_pred,
})
results["Benar"] = results["Label Asli"] == results["Prediksi Model"]

if hasattr(model, "predict_proba"):
    proba = model.predict_proba(X_test)
    max_proba = proba.max(axis=1)
    results["Confidence"] = (max_proba * 100).round(1).astype(str) + "%"

render_scroll_table(results, "Hasil Prediksi Model pada Data Uji")

### Hasil Prediksi Model pada Data Uji

,Label Asli,Prediksi Model,Benar,Confidence
0,Offline Shopper,Online Shopper,False,65.7%
1,Online Shopper,Online Shopper,True,80.2%
2,Online Shopper,Online Shopper,True,96.2%
3,Online Shopper,Online Shopper,True,58.2%


### 3.3 Evaluasi akurasi

In [19]:
accuracy = accuracy_score(y_test, y_pred)
correct_count = int((y_test.values == y_pred).sum())
wrong_count = int(len(y_test) - correct_count)

accuracy_df = pd.DataFrame(
    {
        "Metrik": ["Akurasi", "Jumlah Benar", "Jumlah Salah"],
        "Nilai": [f"{accuracy * 100:.1f}%", str(correct_count), str(wrong_count)],
        "Ringkas": [f"{correct_count} benar dari {len(y_test)} data uji", "-", "-"]
    }
)
render_scroll_table(accuracy_df, "Akurasi Model")

### Akurasi Model

,Metrik,Nilai,Ringkas
0,Akurasi,75.0%,3 benar dari 4 data uji
1,Jumlah Benar,3,-
2,Jumlah Salah,1,-
